In [ ]:
import csv
import json
import os
from collections import defaultdict

DATA_FILE = "habitat_data.json"
DAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
CATEGORIES = ["Social", "Entertainment", "Study", "Productivity", "Games", "Other"]
DISTRACTING = {"Social", "Entertainment", "Games"}

SAMPLE = [
    ("Instagram", "Social", 95, 0), ("YouTube", "Entertainment", 80, 0),
    ("VS Code", "Study", 120, 0), ("Instagram", "Social", 110, 1),
    ("Chrome", "Study", 90, 1), ("Netflix", "Entertainment", 60, 1),
    ("YouTube", "Entertainment", 70, 2), ("Instagram", "Social", 85, 2),
    ("VS Code", "Study", 150, 2), ("Instagram", "Social", 100, 3),
    ("BGMI", "Games", 75, 3), ("Notion", "Productivity", 40, 4),
    ("Instagram", "Social", 120, 4), ("Netflix", "Entertainment", 150, 5),
    ("Instagram", "Social", 140, 5), ("BGMI", "Games", 90, 5),
    ("YouTube", "Entertainment", 120, 6), ("VS Code", "Study", 60, 6),
]

def sample_state():
    return {"goal": 240,
            "entries": [{"app": a, "cat": c, "min": m, "day": d} for a, c, m, d in SAMPLE]}


def load():
    if os.path.exists(DATA_FILE):
        try:
            with open(DATA_FILE, encoding="utf-8") as f:
                return json.load(f)
        except (json.JSONDecodeError, OSError):
            print("Data file could not be read. Starting with sample data.")
    return sample_state()


def save(state):
    with open(DATA_FILE, "w", encoding="utf-8") as f:
        json.dump(state, f, indent=2)


def fmt(m):
    return f"{m // 60}h {m % 60}m" if m >= 60 else f"{m}m"


def ask_int(prompt, lo, hi):
    while True:
        raw = input(prompt).strip()
        if raw.isdigit() and lo <= int(raw) <= hi:
            return int(raw)
        print(f"  Enter a number from {lo} to {hi}.")


def choose(prompt, options):
    for i, o in enumerate(options, 1):
        print(f"  {i}. {o}")
    return ask_int(prompt, 1, len(options)) - 1


def bar(value, maximum, width=30):
    n = round(value / maximum * width) if maximum else 0
    return "#" * n


def analyse(state):
    entries, goal = state["entries"], state["goal"]
    per_day = [0] * 7
    by_app, by_cat = defaultdict(int), defaultdict(int)
    for e in entries:
        per_day[e["day"]] += e["min"]
        by_app[e["app"]] += e["min"]
        by_cat[e["cat"]] += e["min"]
    total = sum(per_day)
    active = sum(1 for v in per_day if v)
    avg = round(total / active) if active else 0
    over = sum(1 for v in per_day if v > goal)

    distract = sum(v for c, v in by_cat.items() if c in DISTRACTING)
    focus = sum(v for c, v in by_cat.items() if c in ("Study", "Productivity"))
    d_share = distract / total if total else 0
    f_share = focus / total if total else 0

    score = 100 - over * 8 - max(0, (avg - goal) / goal) * 30 \
        - max(0, d_share - 0.5) * 60 + f_share * 10
    score = max(0, min(100, round(score))) if total else 0
    return dict(per_day=per_day, by_app=by_app, by_cat=by_cat, total=total,
                avg=avg, over=over, d_share=d_share, f_share=f_share, score=score)


def show_report(state):
    r = analyse(state)
    goal = state["goal"]
    if not r["total"]:
        print("\nNo data yet. Add some entries first.")
        return
    top_app, top_min = max(r["by_app"].items(), key=lambda kv: kv[1])

    print("\n===== WEEKLY SUMMARY =====")
    print(f"Total time      : {fmt(r['total'])}")
    print(f"Daily average   : {fmt(r['avg'])}  (goal {fmt(goal)})")
    print(f"Top app         : {top_app} ({fmt(top_min)})")
    print(f"Days over goal  : {r['over']} of 7")

    print("\n DAILY SCREEN TIME ")
    mx = max(max(r["per_day"]), goal)
    for d, v in zip(DAYS, r["per_day"]):
        flag = " <- over goal" if v > goal else ""
        print(f"{d} | {bar(v, mx):<30} {fmt(v)}{flag}")

    print("\n===== CATEGORY BREAKDOWN =====")
    for c, v in sorted(r["by_cat"].items(), key=lambda kv: -kv[1]):
        print(f"{c:<14}| {bar(v, r['total']):<30} {v / r['total']:.0%}  {fmt(v)}")

    print(f"\n===== HABITAT HEALTH SCORE: {r['score']}/100 =====")
    for tip in insights(r, goal, top_app, top_min):
        print(" -", tip)


def insights(r, goal, top_app, top_min):
    pd, total = r["per_day"], r["total"]
    tips = []
    if r["over"]:
        tips.append(f"You crossed your {fmt(goal)} goal on {r['over']} day(s).")
    else:
        tips.append(f"You stayed within your {fmt(goal)} daily goal all week.")
    worst = pd.index(max(pd))
    tips.append(f"Heaviest day: {DAYS[worst]} ({fmt(pd[worst])}).")
    tips.append(f"{top_app} takes {top_min / total:.0%} of your week. Try an app timer.")
    if r["d_share"] > 0.5:
        tips.append(f"{r['d_share']:.0%} of your time is Social/Entertainment/Games. "
                    "Replace one session a day with study or exercise.")
    else:
        tips.append(f"Study and Productivity make up {r['f_share']:.0%} of your time.")
    weekend, weekday = (pd[5] + pd[6]) / 2, sum(pd[:5]) / 5
    if weekday and weekend > weekday * 1.3:
        tips.append(f"Weekends average {fmt(round(weekend))}, "
                    f"{weekend / weekday - 1:.0%} more than weekdays.")
    return tips



def add_entry(state):
    app = input("App name: ").strip()
    if not app:
        print("  App name cannot be empty.")
        return
    cat = CATEGORIES[choose("Category number: ", CATEGORIES)]
    day = choose("Day number: ", DAYS)
    minutes = ask_int("Minutes (1-1440): ", 1, 1440)
    state["entries"].append({"app": app, "cat": cat, "min": minutes, "day": day})
    save(state)
    print("  Entry added.")


def list_entries(state):
    if not state["entries"]:
        print("\nNo entries.")
        return
    print(f"\n{'#':<4}{'Day':<5}{'App':<15}{'Category':<15}Min")
    for i, e in enumerate(state["entries"], 1):
        print(f"{i:<4}{DAYS[e['day']]:<5}{e['app']:<15}{e['cat']:<15}{e['min']}")


def delete_entry(state):
    list_entries(state)
    if state["entries"]:
        n = ask_int("Entry number to delete: ", 1, len(state["entries"]))
        removed = state["entries"].pop(n - 1)
        save(state)
        print(f"  Deleted {removed['app']}.")


def set_goal(state):
    state["goal"] = ask_int("New daily goal in minutes (30-1440): ", 30, 1440)
    save(state)
    print("  Goal updated.")


def export_csv(state):
    with open("habitat_export.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["day", "app", "category", "minutes"])
        for e in state["entries"]:
            w.writerow([DAYS[e["day"]], e["app"], e["cat"], e["min"]])
    print("  Saved habitat_export.csv")


def main():
    state = load()
    menu = [
        ("Show report", show_report), ("Add entry", add_entry),
        ("List entries", list_entries), ("Delete entry", delete_entry),
        ("Set daily goal", set_goal), ("Export to CSV", export_csv),
        ("Load sample data", None), ("Clear all data", None), ("Quit", None),
    ]
    while True:
        print("\n=== DIGITAL HABITAT ANALYSER ===")
        for i, (name, _) in enumerate(menu, 1):
            print(f"  {i}. {name}")
        pick = ask_int("Choose: ", 1, len(menu))
        name, fn = menu[pick - 1]
        if name == "Quit":
            print("Bye!")
            break
        if name == "Load sample data":
            state = sample_state()
            save(state)
            print("  Sample data loaded.")
        elif name == "Clear all data":
            state["entries"] = []
            save(state)
            print("  All entries cleared.")
        else:
            fn(state)


if __name__ == "__main__":
    main()



=== DIGITAL HABITAT ANALYSER ===
  1. Show report
  2. Add entry
  3. List entries
  4. Delete entry
  5. Set daily goal
  6. Export to CSV
  7. Load sample data
  8. Clear all data
  9. Quit


Choose:  2
App name:  mohsin


  1. Social
  2. Entertainment
  3. Study
  4. Productivity
  5. Games
  6. Other


In [ ]:
import csv
import json
import os
from collections import defaultdict

DATA_FILE = "habitat_data.json"
DAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
CATEGORIES = ["Social", "Entertainment", "Study", "Productivity", "Games", "Other"]
DISTRACTING = {"Social", "Entertainment", "Games"}

SAMPLE = [
    ("Instagram", "Social", 95, 0), ("YouTube", "Entertainment", 80, 0),
    ("VS Code", "Study", 120, 0), ("Instagram", "Social", 110, 1),
    ("Chrome", "Study", 90, 1), ("Netflix", "Entertainment", 60, 1),
    ("YouTube", "Entertainment", 70, 2), ("Instagram", "Social", 85, 2),
    ("VS Code", "Study", 150, 2), ("Instagram", "Social", 100, 3),
    ("BGMI", "Games", 75, 3), ("Notion", "Productivity", 40, 4),
    ("Instagram", "Social", 120, 4), ("Netflix", "Entertainment", 150, 5),
    ("Instagram", "Social", 140, 5), ("BGMI", "Games", 90, 5),
    ("YouTube", "Entertainment", 120, 6), ("VS Code", "Study", 60, 6),
]

def sample_state():
    return {"goal": 240,
            "entries": [{"app": a, "cat": c, "min": m, "day": d} for a, c, m, d in SAMPLE]}


def load():
    if os.path.exists(DATA_FILE):
        try:
            with open(DATA_FILE, encoding="utf-8") as f:
                return json.load(f)
        except (json.JSONDecodeError, OSError):
            print("Data file could not be read. Starting with sample data.")
    return sample_state()


def save(state):
    with open(DATA_FILE, "w", encoding="utf-8") as f:
        json.dump(state, f, indent=2)


def fmt(m):
    return f"{m // 60}h {m % 60}m" if m >= 60 else f"{m}m"


def ask_int(prompt, lo, hi):
    while True:
        raw = input(prompt).strip()
        if raw.isdigit() and lo <= int(raw) <= hi:
            return int(raw)
        print(f"  Enter a number from {lo} to {hi}.")


def choose(prompt, options):
    for i, o in enumerate(options, 1):
        print(f"  {i}. {o}")
    return ask_int(prompt, 1, len(options)) - 1


def bar(value, maximum, width=30):
    n = round(value / maximum * width) if maximum else 0
    return "#" * n


def analyse(state):
    entries, goal = state["entries"], state["goal"]
    per_day = [0] * 7
    by_app, by_cat = defaultdict(int), defaultdict(int)
    for e in entries:
        per_day[e["day"]] += e["min"]
        by_app[e["app"]] += e["min"]
        by_cat[e["cat"]] += e["min"]
    total = sum(per_day)
    active = sum(1 for v in per_day if v)
    avg = round(total / active) if active else 0
    over = sum(1 for v in per_day if v > goal)

    distract = sum(v for c, v in by_cat.items() if c in DISTRACTING)
    focus = sum(v for c, v in by_cat.items() if c in ("Study", "Productivity"))
    d_share = distract / total if total else 0
    f_share = focus / total if total else 0

    score = 100 - over * 8 - max(0, (avg - goal) / goal) * 30 \
        - max(0, d_share - 0.5) * 60 + f_share * 10
    score = max(0, min(100, round(score))) if total else 0
    return dict(per_day=per_day, by_app=by_app, by_cat=by_cat, total=total,
                avg=avg, over=over, d_share=d_share, f_share=f_share, score=score)


def show_report(state):
    r = analyse(state)
    goal = state["goal"]
    if not r["total"]:
        print("\nNo data yet. Add some entries first.")
        return
    top_app, top_min = max(r["by_app"].items(), key=lambda kv: kv[1])

    print("\n===== WEEKLY SUMMARY =====")
    print(f"Total time      : {fmt(r['total'])}")
    print(f"Daily average   : {fmt(r['avg'])}  (goal {fmt(goal)})")
    print(f"Top app         : {top_app} ({fmt(top_min)})")
    print(f"Days over goal  : {r['over']} of 7")

    print("\n DAILY SCREEN TIME ")
    mx = max(max(r["per_day"]), goal)
    for d, v in zip(DAYS, r["per_day"]):
        flag = " <- over goal" if v > goal else ""
        print(f"{d} | {bar(v, mx):<30} {fmt(v)}{flag}")

    print("\n===== CATEGORY BREAKDOWN =====")
    for c, v in sorted(r["by_cat"].items(), key=lambda kv: -kv[1]):
        print(f"{c:<14}| {bar(v, r['total']):<30} {v / r['total']:.0%}  {fmt(v)}")

    print(f"\n===== HABITAT HEALTH SCORE: {r['score']}/100 =====")
    for tip in insights(r, goal, top_app, top_min):
        print(" -", tip)


def insights(r, goal, top_app, top_min):
    pd, total = r["per_day"], r["total"]
    tips = []
    if r["over"]:
        tips.append(f"You crossed your {fmt(goal)} goal on {r['over']} day(s).")
    else:
        tips.append(f"You stayed within your {fmt(goal)} daily goal all week.")
    worst = pd.index(max(pd))
    tips.append(f"Heaviest day: {DAYS[worst]} ({fmt(pd[worst])}).")
    tips.append(f"{top_app} takes {top_min / total:.0%} of your week. Try an app timer.")
    if r["d_share"] > 0.5:
        tips.append(f"{r['d_share']:.0%} of your time is Social/Entertainment/Games. "
                    "Replace one session a day with study or exercise.")
    else:
        tips.append(f"Study and Productivity make up {r['f_share']:.0%} of your time.")
    weekend, weekday = (pd[5] + pd[6]) / 2, sum(pd[:5]) / 5
    if weekday and weekend > weekday * 1.3:
        tips.append(f"Weekends average {fmt(round(weekend))}, "
                    f"{weekend / weekday - 1:.0%} more than weekdays.")
    return tips



def add_entry(state):
    app = input("App name: ").strip()
    if not app:
        print("  App name cannot be empty.")
        return
    cat = CATEGORIES[choose("Category number: ", CATEGORIES)]
    day = choose("Day number: ", DAYS)
    minutes = ask_int("Minutes (1-1440): ", 1, 1440)
    state["entries"].append({"app": app, "cat": cat, "min": minutes, "day": day})
    save(state)
    print("  Entry added.")


def list_entries(state):
    if not state["entries"]:
        print("\nNo entries.")
        return
    print(f"\n{'#':<4}{'Day':<5}{'App':<15}{'Category':<15}Min")
    for i, e in enumerate(state["entries"], 1):
        print(f"{i:<4}{DAYS[e['day']]:<5}{e['app']:<15}{e['cat']:<15}{e['min']}")


def delete_entry(state):
    list_entries(state)
    if state["entries"]:
        n = ask_int("Entry number to delete: ", 1, len(state["entries"]))
        removed = state["entries"].pop(n - 1)
        save(state)
        print(f"  Deleted {removed['app']}.")


def set_goal(state):
    state["goal"] = ask_int("New daily goal in minutes (30-1440): ", 30, 1440)
    save(state)
    print("  Goal updated.")


def export_csv(state):
    with open("habitat_export.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["day", "app", "category", "minutes"])
        for e in state["entries"]:
            w.writerow([DAYS[e["day"]], e["app"], e["cat"], e["min"]])
    print("  Saved habitat_export.csv")


def main():
    state = load()
    menu = [
        ("Show report", show_report), ("Add entry", add_entry),
        ("List entries", list_entries), ("Delete entry", delete_entry),
        ("Set daily goal", set_goal), ("Export to CSV", export_csv),
        ("Load sample data", None), ("Clear all data", None), ("Quit", None),
    ]
    while True:
        print("\n=== DIGITAL HABITAT ANALYSER ===")
        for i, (name, _) in enumerate(menu, 1):
            print(f"  {i}. {name}")
        pick = ask_int("Choose: ", 1, len(menu))
        name, fn = menu[pick - 1]
        if name == "Quit":
            print("Bye!")
            break
        if name == "Load sample data":
            state = sample_state()
            save(state)
            print("  Sample data loaded.")
        elif name == "Clear all data":
            state["entries"] = []
            save(state)
            print("  All entries cleared.")
        else:
            fn(state)


if __name__ == "__main__":
    main()